# FAIR²-FoodHealth Exploration with `mlcroissant`
This notebook demonstrates how to load, inspect, and explore a Croissant-described dataset using the `mlcroissant` library.

It follows a step-by-step workflow:
- load the dataset from its Croissant JSON-LD URL,
- inspect metadata and available record sets,
- load records into pandas DataFrames,
- perform lightweight exploratory data analysis (EDA), and
- visualize useful summaries.

### Dataset Source
Croissant schema URL:
`https://sen.science/doi/10.71728/senscience.3yz2-mb2d/fair2.json`

### Important note on identifiers
In Croissant, record sets, fields, file objects, and related entities are identified by `@id`. In this notebook, we always inspect and refer to dataset entities using their `@id` values when available.

### Dataset note
The FAIR²-FoodHealth package description indicates a provenance-rich, multi-table dataset related to foods, nutrients, conditions, mappings, sources, and governance. Some tables may be synthetic, curated, empirical, or review-oriented. Interpret analysis results accordingly.

In [ ]:
# Ensure required libraries are installed
!pip install -q mlcroissant pandas matplotlib seaborn

## 1. Data Loading
Load the dataset from the Croissant schema URL and inspect top-level metadata.

A key implementation detail is that `dataset.metadata` is treated as a single metadata object. We convert it to JSON only when we want a serializable representation for inspection.

In [ ]:
import json
import warnings

import mlcroissant as mlc
import pandas as pd

warnings.filterwarnings("ignore")

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.3yz2-mb2d/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata as a single object, then convert to JSON for inspection
metadata_json = dataset.metadata.to_json()

print("Dataset loaded successfully.")
print(f"Name: {metadata_json.get('name', 'N/A')}")
print(f"Version: {metadata_json.get('version', 'N/A')}")
print(f"License: {metadata_json.get('license', 'N/A')}")
print(f"Identifier: {metadata_json.get('identifier', 'N/A')}")
print("\nDescription:")
print(metadata_json.get('description', 'N/A'))

In [ ]:
# Show selected high-level metadata fields
summary_fields = [
    "name",
    "identifier",
    "version",
    "datePublished",
    "license",
    "keywords",
    "citeAs",
]

summary = {field: metadata_json.get(field) for field in summary_fields}
pd.Series(summary)

## 2. Data Overview
Review the available record sets and inspect their fields and `@id` values.

Because Croissant datasets may expose many tables, we first build helper functions to safely inspect the metadata structure. The exact schema object layout can vary by dataset and library version, so the code below is intentionally defensive.

In [ ]:
def _as_list(value):
    if value is None:
        return []
    return value if isinstance(value, list) else [value]


def get_recordset_metadata(metadata_dict):
    """Return record set metadata entries from the metadata JSON representation."""
    return _as_list(metadata_dict.get("recordSet", []))


def extract_field_entries(record_set_entry):
    """Extract field entries from a record set JSON entry."""
    return _as_list(record_set_entry.get("field", []))


recordset_entries = get_recordset_metadata(metadata_json)
print(f"Number of record sets declared in metadata JSON: {len(recordset_entries)}")

In [ ]:
# Build a record set overview table using @id values whenever present
recordset_rows = []

for rs in recordset_entries:
    rs_id = rs.get("@id", "N/A")
    rs_name = rs.get("name", "")
    rs_desc = rs.get("description", "")
    rs_fields = extract_field_entries(rs)
    recordset_rows.append(
        {
            "record_set_id": rs_id,
            "record_set_name": rs_name,
            "field_count": len(rs_fields),
            "description": rs_desc,
        }
    )

recordset_df = pd.DataFrame(recordset_rows)
recordset_df

If the dataset exposes record sets through Croissant metadata, the next cell expands them and lists field-level identifiers.

In [ ]:
field_rows = []

for rs in recordset_entries:
    rs_id = rs.get("@id", "N/A")
    rs_name = rs.get("name", "")
    for field in extract_field_entries(rs):
        field_rows.append(
            {
                "record_set_id": rs_id,
                "record_set_name": rs_name,
                "field_id": field.get("@id", "N/A"),
                "field_name": field.get("name", ""),
                "data_type": field.get("dataType", ""),
                "description": field.get("description", ""),
            }
        )

fields_df = pd.DataFrame(field_rows)
fields_df.head(20)

### Preview records from a record set
When record sets are available, we can preview records using the record set `@id`.

The cell below dynamically chooses the first available record set `@id` and prints a few records.

In [ ]:
available_record_set_ids = recordset_df["record_set_id"].dropna().tolist() if not recordset_df.empty else []

example_record_set_id = available_record_set_ids[0] if available_record_set_ids else None
print("Example record set @id:", example_record_set_id)

if example_record_set_id is not None:
    try:
        for idx, record in enumerate(dataset.records(record_set=example_record_set_id)):
            print(record)
            if idx >= 2:
                break
    except Exception as e:
        print("Could not preview records from the selected record set.")
        print(type(e).__name__, str(e))
else:
    print("No record sets were declared in metadata JSON, so there is nothing to preview here.")

## 3. Data Extraction
Load data from one or more record sets into pandas DataFrames.

This section uses variables so the workflow adapts automatically to the dataset structure. Each table is stored in a dictionary keyed by the record set `@id`.

In [ ]:
# Select record sets dynamically
record_set_ids = available_record_set_ids.copy()
dataframes = {}
load_errors = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
    except Exception as e:
        load_errors[record_set_id] = f"{type(e).__name__}: {e}"

print(f"Loaded {len(dataframes)} record set(s) into DataFrames.")
if load_errors:
    print("\nRecord sets that could not be loaded:")
    for k, v in load_errors.items():
        print(f"- {k}: {v}")

In [ ]:
# Show shapes for loaded DataFrames
df_overview_rows = []

for record_set_id, df in dataframes.items():
    df_overview_rows.append(
        {
            "record_set_id": record_set_id,
            "rows": df.shape[0],
            "columns": df.shape[1],
            "column_names": ", ".join(df.columns.astype(str).tolist()[:10]) + (" ..." if df.shape[1] > 10 else ""),
        }
    )

df_overview = pd.DataFrame(df_overview_rows).sort_values(["rows", "columns"], ascending=False) if df_overview_rows else pd.DataFrame()
df_overview

In [ ]:
# Inspect one loaded DataFrame
selected_record_set_id = next(iter(dataframes.keys()), None)
print("Selected record set @id:", selected_record_set_id)

if selected_record_set_id is not None:
    print("\nColumns:")
    print(dataframes[selected_record_set_id].columns.tolist())
    print("\nHead:")
    display(dataframes[selected_record_set_id].head())
else:
    print("No DataFrames are available to inspect.")

### Optional: inspect all loaded tables
This helper cell provides a compact summary of loaded record sets and their columns.

In [ ]:
for record_set_id, df in dataframes.items():
    print("=" * 100)
    print(f"Record set @id: {record_set_id}")
    print(f"Shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    print()
    display(df.head(3))

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps to a selected record set.

Because this is a general-purpose Croissant notebook, we detect usable numeric and categorical columns automatically. This makes the notebook robust across different dataset table structures.

In [ ]:
import numpy as np

def pick_analysis_dataframe(dataframe_dict):
    """Pick the largest non-empty DataFrame for analysis."""
    non_empty = [(rs_id, df) for rs_id, df in dataframe_dict.items() if not df.empty]
    if not non_empty:
        return None, None
    non_empty.sort(key=lambda x: (x[1].shape[0], x[1].shape[1]), reverse=True)
    return non_empty[0]


analysis_record_set_id, analysis_df = pick_analysis_dataframe(dataframes)
print("Analysis record set @id:", analysis_record_set_id)
print("Shape:" , None if analysis_df is None else analysis_df.shape)

In [ ]:
if analysis_df is not None:
    working_df = analysis_df.copy()
    
    # Try to coerce object columns to numeric when possible
    for col in working_df.columns:
        if working_df[col].dtype == "object":
            try:
                converted = pd.to_numeric(working_df[col], errors="ignore")
                working_df[col] = converted
            except Exception:
                pass
    
    numeric_columns = working_df.select_dtypes(include=[np.number]).columns.tolist()
    categorical_columns = [c for c in working_df.columns if c not in numeric_columns]
    
    print("Numeric columns:", numeric_columns)
    print("\nCategorical/non-numeric columns:", categorical_columns[:20])
else:
    working_df = None
    numeric_columns = []
    categorical_columns = []
    print("No analysis DataFrame is available.")

In [ ]:
# Summary statistics for numeric columns
if working_df is not None and numeric_columns:
    display(working_df[numeric_columns].describe().T)
else:
    print("No numeric columns available for summary statistics.")

In [ ]:
# Missingness overview
if working_df is not None:
    missing_df = pd.DataFrame({
        "column": working_df.columns,
        "missing_count": working_df.isna().sum().values,
        "missing_pct": (working_df.isna().mean().values * 100).round(2),
        "dtype": working_df.dtypes.astype(str).values,
    }).sort_values("missing_pct", ascending=False)
    display(missing_df.head(20))
else:
    print("No DataFrame available for missingness analysis.")

### Filtering, normalization, and grouping
The next cell mirrors the template workflow using dynamically selected columns.

- A numeric field is chosen automatically.
- A threshold is defined from the distribution.
- Records above the threshold are filtered.
- A normalized version of the numeric field is created.
- If a categorical grouping field exists, group-level statistics are computed.

In [ ]:
if working_df is not None and numeric_columns:
    numeric_field = numeric_columns[0]
    threshold = working_df[numeric_field].median()
    filtered_df = working_df[working_df[numeric_field] > threshold].copy()
    
    print(f"Selected numeric field: {numeric_field}")
    print(f"Threshold (median): {threshold}")
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())
    
    std_value = filtered_df[numeric_field].std()
    if pd.notna(std_value) and std_value != 0:
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / std_value
        )
    else:
        filtered_df[f"{numeric_field}_normalized"] = 0.0
    
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    
    group_field = categorical_columns[0] if categorical_columns else None
    if group_field is not None and group_field in filtered_df.columns:
        grouped_df = (
            filtered_df.groupby(group_field)[numeric_field]
            .agg(["count", "mean", "median", "min", "max"])
            .sort_values("count", ascending=False)
        )
        print(f"Grouped data by {group_field}:")
        display(grouped_df.head(10))
    else:
        grouped_df = None
        print("No suitable grouping field was found.")
else:
    numeric_field = None
    threshold = None
    filtered_df = None
    grouped_df = None
    group_field = None
    print("EDA filtering and normalization skipped because no numeric columns are available.")

In [ ]:
# Simple outlier handling example using the IQR rule
if working_df is not None and numeric_columns:
    outlier_field = numeric_columns[0]
    q1 = working_df[outlier_field].quantile(0.25)
    q3 = working_df[outlier_field].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    
    no_outliers_df = working_df[
        (working_df[outlier_field] >= lower_bound) &
        (working_df[outlier_field] <= upper_bound)
    ].copy()
    
    print(f"Outlier analysis for: {outlier_field}")
    print(f"Lower bound: {lower_bound}")
    print(f"Upper bound: {upper_bound}")
    print(f"Original rows: {len(working_df)}")
    print(f"Rows after outlier removal: {len(no_outliers_df)}")
else:
    no_outliers_df = None
    print("Outlier analysis skipped because no numeric columns are available.")

## 5. Visualization
Visualize distributions and simple grouped summaries for the selected record set.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

In [ ]:
# Histogram for the selected numeric field
if working_df is not None and numeric_field is not None:
    plt.figure(figsize=(10, 5))
    sns.histplot(working_df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}\nRecord set: {analysis_record_set_id}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()
else:
    print("Histogram skipped because no numeric field is available.")

In [ ]:
# Boxplot for outlier inspection
if working_df is not None and numeric_field is not None:
    plt.figure(figsize=(10, 2.5))
    sns.boxplot(x=working_df[numeric_field].dropna())
    plt.title(f"Boxplot of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.tight_layout()
    plt.show()
else:
    print("Boxplot skipped because no numeric field is available.")

In [ ]:
# Bar plot for grouped summary if a categorical grouping exists
if grouped_df is not None and group_field is not None and not grouped_df.empty:
    top_grouped = grouped_df.head(10).reset_index()
    plt.figure(figsize=(12, 5))
    sns.barplot(data=top_grouped, x=group_field, y="mean")
    plt.title(f"Mean {numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(f"Mean {numeric_field}")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("Grouped bar chart skipped because no grouped summary is available.")

In [ ]:
# Correlation heatmap when multiple numeric columns exist
if working_df is not None and len(numeric_columns) >= 2:
    corr = working_df[numeric_columns].corr(numeric_only=True)
    plt.figure(figsize=(8, 6))
    sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
    plt.title("Correlation heatmap for numeric fields")
    plt.tight_layout()
    plt.show()
else:
    print("Correlation heatmap skipped because fewer than two numeric columns are available.")

## 6. Conclusion
This notebook showed how to:

- load a Croissant dataset using `mlcroissant`,
- inspect dataset metadata as a single metadata object,
- review available record sets and fields using their `@id` values,
- load records into pandas DataFrames,
- perform basic filtering, normalization, missingness checks, and outlier handling, and
- create simple visualizations for numeric and grouped summaries.

### Interpretation reminder
The FAIR²-FoodHealth dataset package combines multiple data layers, including empirical, curated, synthetic, provenance, and governance-oriented content. Before downstream modeling or inference, separate tables by purpose and pay close attention to fields related to review state, synthetic status, provenance, and interpretation boundaries.

### Next steps
Potential next analyses include:
- joining related record sets by identifier fields,
- isolating empirical food composition tables,
- analyzing semantic mapping coverage,
- auditing provenance completeness, and
- building task-specific subsets for nutrition, guideline navigation, or ontology alignment.